# CV Lab Exam — Function Library (Labs 1–6)

**How to use:** run the *Setup* cell first, then run every section cell once (each only defines functions, so this takes about a second).
After that, any task is a few calls, e.g. `show([gray, equalize(gray)], ['Original', 'Equalized'])`.
The last cell, *Self-test*, runs every function on synthetic images — use it to check the environment on exam day.

**Conventions used everywhere:** images are NumPy arrays; OpenCV colour order is **BGR**; points are **(x, y)**; slicing is **[y, x]**; sizes passed to `resize`/`warp*` are **(width, height)**.

## 0. Setup — imports, display, loading, practice images

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import deque

try:                                    # Lab 4 features (installed in the exam lab)
    from skimage.feature import hog, local_binary_pattern
except ImportError:
    hog = local_binary_pattern = None
try:                                    # Lab 6 wavelets
    import pywt
except ImportError:
    pywt = None

print('OpenCV', cv2.__version__, '| skimage', hog is not None, '| pywt', pywt is not None)


def show(imgs, titles=None, cols=3, cmap='gray'):
    """Display a list of images in a grid. Colour (BGR) images are converted to RGB; gray uses cmap."""
    if not isinstance(imgs, (list, tuple)):
        imgs = [imgs]
    titles = titles or [''] * len(imgs)
    rows = (len(imgs) + cols - 1) // cols
    plt.figure(figsize=(5 * cols, 4 * rows))
    for i, (im, t) in enumerate(zip(imgs, titles)):
        plt.subplot(rows, cols, i + 1)
        if im.ndim == 3:
            plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        else:
            plt.imshow(im, cmap=cmap)
        plt.title(t)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


def load(path, gray=False):
    """Read an image safely: raises FileNotFoundError instead of returning None silently."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f'Error: image not found -> {path}')
    return img


def to_gray(img):
    """Return a 1-channel image whether the input is gray or BGR."""
    return img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def to_bgr(img):
    """Return a 3-channel image whether the input is gray or BGR (needed for hstack, watershed, drawing in colour)."""
    return img if img.ndim == 3 else cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)


# ---- Practice images (stand-ins when the exam image is missing) ----
def make_coins(touching=True):
    """Gray image: bright coins on a dark background; touching=True makes some coins touch."""
    img = np.full((400, 500), 40, np.uint8)
    centres = [(100, 100), (170, 110), (320, 120), (120, 280), (260, 260), (330, 290), (420, 300)]
    if not touching:
        centres = [(100, 100), (250, 100), (400, 100), (100, 280), (250, 280), (400, 280)]
    for c in centres:
        cv2.circle(img, c, 45, 200, -1)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    noise = np.random.default_rng(0).normal(0, 8, img.shape)
    return np.uint8(np.clip(img + noise, 0, 255))


def make_document():
    """Gray text page with uneven lighting (dark left, bright right), like sudoku.png."""
    page = np.full((300, 500), 255, np.uint8)
    for r in range(6):
        cv2.putText(page, 'CV LAB EXAM 2026', (20, 40 + r * 45), cv2.FONT_HERSHEY_SIMPLEX, 1.2, 0, 2)
    ramp = np.tile(np.linspace(0.35, 1.0, 500), (300, 1))
    return np.uint8(page * ramp)


def make_colour_objects():
    """BGR image of coloured discs (red, yellow, blue, green, orange, purple), like smarties.png."""
    img = np.full((300, 450, 3), 230, np.uint8)
    cols = [(0, 0, 255), (0, 255, 255), (255, 0, 0), (0, 200, 0), (0, 140, 255), (200, 0, 200)]
    for i, c in enumerate(cols):
        cv2.circle(img, (60 + i * 65, 150), 28, c, -1)
    return img


def make_dark_xray():
    """Under-exposed gray image with intensities squeezed into 0-80 (for Lab 2 enhancement)."""
    base = np.zeros((300, 300), np.uint8)
    cv2.ellipse(base, (100, 150), (50, 100), 0, 0, 360, 255, -1)
    cv2.ellipse(base, (200, 150), (50, 100), 0, 0, 360, 255, -1)
    for y in range(40, 280, 25):
        cv2.line(base, (40, y), (260, y), 120, 6)
    base = cv2.GaussianBlur(base, (31, 31), 0)
    return np.uint8(base.astype(np.float64) * 80 / 255)    # float first: uint8 * 80 would wrap


def make_textured(h=400, w=600, seed=1):
    """BGR scene of random shapes: lots of corners, so SIFT finds keypoints."""
    rng = np.random.default_rng(seed)
    img = np.full((h, w, 3), 200, np.uint8)
    for _ in range(80):
        c = tuple(int(v) for v in rng.integers(0, 255, 3))
        p = (int(rng.integers(0, w)), int(rng.integers(0, h)))
        if rng.random() < 0.5:
            cv2.circle(img, p, int(rng.integers(5, 30)), c, -1)
        else:
            cv2.rectangle(img, p, (p[0] + int(rng.integers(10, 50)), p[1] + int(rng.integers(10, 50))), c, -1)
    return img


def make_road():
    """BGR road image with two white lane lines (for HoughLinesP)."""
    road = np.full((360, 640, 3), 60, np.uint8)
    cv2.line(road, (120, 359), (300, 200), (255, 255, 255), 8)
    cv2.line(road, (560, 359), (360, 200), (255, 255, 255), 8)
    return road


def make_grid_img(h=300, w=400):
    """BGR grid with a box and text, so geometric warps are easy to see."""
    img = np.full((h, w, 3), 255, np.uint8)
    for x in range(0, w, 40):
        cv2.line(img, (x, 0), (x, h), (200, 200, 200), 1)
    for y in range(0, h, 40):
        cv2.line(img, (0, y), (w, y), (200, 200, 200), 1)
    cv2.rectangle(img, (w // 2 - 80, h // 2 - 60), (w // 2 + 80, h // 2 + 60), (255, 0, 0), 3)
    cv2.putText(img, 'CV', (w // 2 - 35, h // 2 + 20), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
    return img


def make_sensor_signal(n=1000, seed=42):
    """1-D noisy sine wave with 5 injected spikes. Returns (signal, true_anomaly_indices)."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    sig = np.sin(2 * np.pi * t / 200) + 0.2 * rng.normal(size=n)
    idx = [150, 400, 401, 650, 880]
    sig[idx] += [3, -3, 3, 2.5, -2.8]
    return sig, idx

## 1. Lab 1 — Basics: display, ROI, drawing, blending, masks, statistics

In [ ]:
def show_rgb(img, title='', title_kw=None, figsize=(8, 6)):
    """Show one BGR image correctly in Matplotlib with a formatted title and no axes."""
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap='gray')
    plt.title(title, **(title_kw or {'fontsize': 16, 'color': 'darkred', 'pad': 15}))
    plt.axis('off')
    plt.show()


def resize(img, width=None, height=None, fx=None, fy=None):
    """Resize by exact size (width, height) or by factors fx, fy. Note OpenCV order is (W, H)."""
    if fx is not None:
        return cv2.resize(img, None, fx=fx, fy=fy if fy is not None else fx)
    return cv2.resize(img, (width, height))


def crop_center(img, size_w, size_h=None):
    """Extract a size_w x size_h ROI from the exact centre using slicing [y, x]."""
    size_h = size_h or size_w
    h, w = img.shape[:2]
    y0, x0 = h // 2 - size_h // 2, w // 2 - size_w // 2
    return img[y0:y0 + size_h, x0:x0 + size_w]


def gaussian_blur(img, k=25):
    """Gaussian blur with a k x k kernel (k must be odd; sigma 0 = computed from k)."""
    return cv2.GaussianBlur(img, (k, k), 0)


def draw_target(size=800, n=5, colors=((0, 0, 255), (255, 255, 255))):
    """Black canvas with n filled concentric circles of alternating colour + bounding box. Returns (canvas, centre)."""
    canvas = np.zeros((size, size, 3), np.uint8)
    c = (size // 2, size // 2)                                   # (x, y)
    r_max = int(size * 0.3125)
    radii = np.linspace(r_max, r_max / n, n).astype(int)        # big -> small so all stay visible
    for i, r in enumerate(radii):
        cv2.circle(canvas, c, int(r), colors[i % 2], -1)         # thickness -1 = filled
    cv2.rectangle(canvas, (c[0] - r_max, c[1] - r_max), (c[0] + r_max, c[1] + r_max), (0, 255, 0), 3)
    return canvas, c


def caption_overlay(img, text, frac=0.2, color=(255, 0, 0), alpha=0.4):
    """Semi-transparent coloured band over the bottom `frac` of the image with text (addWeighted blend)."""
    h, w = img.shape[:2]
    overlay = img.copy()
    cv2.rectangle(overlay, (0, int(h * (1 - frac))), (w, h), color, -1)
    out = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)     # alpha*overlay + (1-alpha)*img
    cv2.putText(out, text, (20, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    return out


def mask_combine(img_a, img_b, size=500, radius=150):
    """Cut a circle from img_a and the background from img_b, then combine (bitwise and/not/or).
    Returns (mask, foreground, background, combined)."""
    a, b = cv2.resize(img_a, (size, size)), cv2.resize(img_b, (size, size))   # sizes must match
    mask = np.zeros((size, size), np.uint8)
    cv2.circle(mask, (size // 2, size // 2), radius, 255, -1)
    fg = cv2.bitwise_and(a, a, mask=mask)
    bg = cv2.bitwise_and(b, b, mask=cv2.bitwise_not(mask))
    return mask, fg, bg, cv2.bitwise_or(fg, bg)


def channel_stats(img):
    """Flatten B, G, R into a Pandas DataFrame (columns Red, Green, Blue) and return (df, df.describe())."""
    b, g, r = cv2.split(img)
    df = pd.DataFrame({'Red': r.flatten(), 'Green': g.flatten(), 'Blue': b.flatten()})
    return df, df.describe()


def rotate(img, angle, scale=1.0):
    """Rotate about the centre with getRotationMatrix2D (+angle = counter-clockwise)."""
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    return cv2.warpAffine(img, M, (w, h))

## 2. Lab 2 — Point / photometric transforms and histograms
A point transform maps each pixel independently: s = T(r). Fastest implementation = a 256-entry lookup table (LUT).

In [ ]:
def negative(g):
    """Image negative s = 255 - r. Reveals detail in dark regions."""
    return 255 - g


def log_transform(g):
    """Log s = c*log(1+r), c = 255/log(1+r_max). Expands DARK values, compresses bright ones."""
    f = g.astype(np.float64)
    c = 255 / np.log(1 + f.max())
    return np.uint8(np.clip(c * np.log1p(f), 0, 255))


def inverse_log_transform(g):
    """Inverse log s = exp(r/c) - 1. Expands BRIGHT values (opposite of log)."""
    c = 255 / np.log(256)
    return np.uint8(np.clip(np.expm1(g.astype(np.float64) / c), 0, 255))


def gamma_transform(g, gamma):
    """Power law s = 255*(r/255)^gamma. gamma<1 brightens, gamma>1 darkens, gamma=1 identity. Works on colour too."""
    table = np.array([255 * (i / 255) ** gamma for i in range(256)]).astype(np.uint8)
    return cv2.LUT(g, table)


def contrast_stretch(g, r1, s1, r2, s2):
    """Piecewise-linear stretch through (0,0),(r1,s1),(r2,s2),(255,255). r1=r2, s1=0, s2=255 -> thresholding."""
    table = np.interp(np.arange(256), [0, r1, r2, 255], [0, s1, s2, 255]).astype(np.uint8)
    return cv2.LUT(g, table)


def minmax_stretch(g):
    """Min-max stretch s = (r - r_min)*255/(r_max - r_min): uses the image's own range."""
    return cv2.normalize(g, None, 0, 255, cv2.NORM_MINMAX)


def slice_levels(g, A, B, keep_background=True):
    """Intensity-level slicing: pixels in [A, B] -> 255; others kept (or set to 0 if keep_background=False)."""
    out = g.copy() if keep_background else np.zeros_like(g)
    out[(g >= A) & (g <= B)] = 255
    return out


def bit_plane(g, k):
    """Bit-plane k (0 = least significant, 7 = most significant) as a 0/255 image."""
    return ((g >> k) & 1) * 255


def plot_hist(img, title='Histogram'):
    """Plot the histogram: one curve for gray, three (b, g, r) for colour."""
    plt.figure(figsize=(6, 3))
    if img.ndim == 2:
        plt.plot(cv2.calcHist([img], [0], None, [256], [0, 256]), color='k')
    else:
        for i, col in enumerate('bgr'):
            plt.plot(cv2.calcHist([img], [i], None, [256], [0, 256]), color=col)
    plt.xlim([0, 256]); plt.title(title); plt.xlabel('intensity'); plt.ylabel('count')
    plt.show()


def equalize(g):
    """Histogram equalization (built-in): s_k = 255 * CDF(r_k). Gray uint8 only, no parameters."""
    return cv2.equalizeHist(g)


def equalize_manual(g):
    """Histogram equalization from scratch via the CDF (use if asked not to call equalizeHist)."""
    hist, _ = np.histogram(g.flatten(), 256, [0, 256])
    cdf = hist.cumsum()
    cdf_m = np.ma.masked_equal(cdf, 0)
    cdf_m = (cdf_m - cdf_m.min()) * 255 / (cdf_m.max() - cdf_m.min())
    return np.ma.filled(cdf_m, 0).astype(np.uint8)[g]


def clahe(g, clip=2.0, tiles=8):
    """CLAHE: equalization per tile with a clip limit so noise is not amplified."""
    return cv2.createCLAHE(clipLimit=clip, tileGridSize=(tiles, tiles)).apply(g)


def equalize_colour(img):
    """Equalize a colour image on the luminance (Y of YCrCb) only, so colours do not shift."""
    ycc = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    ycc[:, :, 0] = cv2.equalizeHist(ycc[:, :, 0])
    return cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR)


def compare_with_hist(imgs, titles):
    """Row of gray images with their histograms underneath (before/after figure)."""
    n = len(imgs)
    plt.figure(figsize=(4 * n, 7))
    for i, (im, t) in enumerate(zip(imgs, titles)):
        plt.subplot(2, n, i + 1); plt.imshow(im, cmap='gray'); plt.title(t); plt.axis('off')
        plt.subplot(2, n, i + n + 1); plt.hist(im.ravel(), bins=256, range=(0, 256))
    plt.tight_layout(); plt.show()


def plot_transfer_curves(gammas=(0.4, 2.5), pw=((70, 20), (180, 235))):
    """Plot s = T(r) for identity, negative, log, gammas and a piecewise stretch on one graph."""
    r = np.arange(256)
    plt.figure(figsize=(6, 6))
    plt.plot(r, r, 'k--', label='identity')
    plt.plot(r, 255 - r, label='negative')
    plt.plot(r, (255 / np.log(256)) * np.log1p(r), label='log')
    for gm in gammas:
        plt.plot(r, 255 * (r / 255) ** gm, label=f'gamma {gm}')
    (r1, s1), (r2, s2) = pw
    plt.plot(r, np.interp(r, [0, r1, r2, 255], [0, s1, s2, 255]), label=f'piecewise ({r1},{s1}),({r2},{s2})')
    plt.xlabel('input r'); plt.ylabel('output s'); plt.legend(); plt.show()


def colormap(g, cmap=cv2.COLORMAP_JET):
    """False-colour heatmap from a 1-channel uint8 image (JET: blue = low, red = high)."""
    return cv2.applyColorMap(g, cmap)


def gray_world(bgr):
    """Gray-world colour balance: scale each channel so its mean equals the overall mean."""
    f = bgr.astype(np.float64)
    avg = f.reshape(-1, 3).mean(axis=0)
    f *= avg.mean() / avg
    return np.uint8(np.clip(f, 0, 255))


def brightness_contrast(img, alpha=1.2, beta=20):
    """Linear point transform s = alpha*r + beta (alpha = contrast, beta = brightness), saturated to 0-255."""
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)


def dense_mask(g, T=200):
    """Keep only pixels brighter than T (e.g. densest tissue on an X-ray). Returns (mask, masked image)."""
    _, m = cv2.threshold(g, T, 255, cv2.THRESH_BINARY)
    return m, cv2.bitwise_and(g, g, mask=m)


def fuse(ct, mri, w_ct=0.7, w_mri=0.3):
    """CT + MRI fusion: equalize both, colour-map the MRI, weighted sum (heavier weight = dominant modality)."""
    mri = cv2.resize(mri, (ct.shape[1], ct.shape[0]))
    ct_c = cv2.cvtColor(cv2.equalizeHist(ct), cv2.COLOR_GRAY2BGR)
    mri_c = cv2.applyColorMap(cv2.equalizeHist(mri), cv2.COLORMAP_JET)
    return cv2.addWeighted(ct_c, w_ct, mri_c, w_mri, 0)


def enhance_frame(frame, gamma=1.5):
    """Lab 2 video pipeline for one frame: equalize -> log -> gamma -> JET -> gray-world."""
    g = cv2.equalizeHist(to_gray(frame))
    g = gamma_transform(log_transform(g), gamma)
    return gray_world(cv2.applyColorMap(g, cv2.COLORMAP_JET))


def process_video(path, func=enhance_frame, max_frames=None, use_imshow=True):
    """Run func on every frame and show raw | processed side by side. Press q to quit (imshow mode).
    In Jupyter set use_imshow=False: returns a list of combined frames to pass to show()."""
    cap = cv2.VideoCapture(path)                     # 0 = webcam
    saved, i = [], 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break                                     # end of video
        combined = np.hstack([frame, to_bgr(func(frame))])   # same height & channels needed
        if use_imshow:
            cv2.imshow('Raw | Enhanced', combined)
            if cv2.waitKey(25) & 0xFF == ord('q'):
                break
        elif i % 30 == 0:
            saved.append(combined)
        i += 1
        if max_frames and i >= max_frames:
            break
    cap.release()
    if use_imshow:
        cv2.destroyAllWindows()
    return saved

## 3. Lab 3 — Geometric transforms (manual matrices, homogeneous coordinates)
3x3 matrices compose by `@`; the **rightmost** matrix is applied first. Image y points down, so `R(+deg)` turns clockwise on screen.

In [ ]:
def T(tx, ty):
    """Translation matrix (3x3): +x right, +y down."""
    return np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]], np.float64)


def S(sx, sy=None):
    """Scale matrix (3x3) about the origin."""
    sy = sx if sy is None else sy
    return np.array([[sx, 0, 0], [0, sy, 0], [0, 0, 1]], np.float64)


def Sh(shx=0.0, shy=0.0):
    """Shear matrix (3x3): x' = x + shx*y, y' = y + shy*x."""
    return np.array([[1, shx, 0], [shy, 1, 0], [0, 0, 1]], np.float64)


def R(deg):
    """Rotation matrix (3x3), textbook form. On screen +deg = clockwise; use R(-deg) for anticlockwise."""
    t = np.deg2rad(deg)
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], np.float64)


def about_center(M, w, h):
    """Make M act about the image centre: T(c) @ M @ T(-c)."""
    return T(w / 2, h / 2) @ M @ T(-w / 2, -h / 2)


def apply(img, M3, size=None):
    """Warp with a 3x3 AFFINE matrix (uses top 2 rows). size = (W, H), default = input size."""
    h, w = img.shape[:2]
    return cv2.warpAffine(img, M3[:2].astype(np.float32), size or (w, h))


def scale_centered(img, sx, sy=None):
    """Scale about the centre on the same canvas (zoom keeps the object centred)."""
    h, w = img.shape[:2]
    return apply(img, about_center(S(sx, sy), w, h))


def rotate_no_clip(img, deg):
    """Rotate anticlockwise by deg with a manual matrix and an enlarged canvas so no corner is cut."""
    h, w = img.shape[:2]
    c, s = abs(np.cos(np.deg2rad(deg))), abs(np.sin(np.deg2rad(deg)))
    nw, nh = int(np.ceil(w * c + h * s)), int(np.ceil(w * s + h * c))
    M = T(nw / 2, nh / 2) @ R(-deg) @ T(-w / 2, -h / 2)
    return apply(img, M, (nw, nh))


def shear_no_clip(img, shx):
    """Horizontal shear with the canvas widened by |shx|*h so nothing is cut (flip sign to undo a slant)."""
    h, w = img.shape[:2]
    pad = int(abs(shx) * h)
    M = (T(pad, 0) if shx < 0 else T(0, 0)) @ Sh(shx)
    return apply(img, M, (w + pad, h))


def rigid(deg, tx, ty, w, h):
    """Rigid (Euclidean) matrix: rotate about centre then translate. Keeps lengths and angles (3 DOF)."""
    return T(tx, ty) @ about_center(R(-deg), w, h)


def similarity(scale, deg, tx, ty, w, h):
    """Similarity matrix: scale + rotate about centre + translate. Keeps angles and shape (4 DOF)."""
    return T(tx, ty) @ about_center(R(-deg) @ S(scale), w, h)


def solve_affine(src3, dst3):
    """Solve the 6 affine unknowns from 3 point pairs by linear equations (manual getAffineTransform). Returns 2x3."""
    A, B = [], []
    for (x, y), (xp, yp) in zip(src3, dst3):
        A.append([x, y, 1, 0, 0, 0]); B.append(xp)
        A.append([0, 0, 0, x, y, 1]); B.append(yp)
    return np.linalg.solve(np.array(A, float), np.array(B, float)).reshape(2, 3)


def birdseye(img, corners4, out_w, out_h):
    """Perspective warp: 4 corners (TL, TR, BR, BL) -> out_w x out_h rectangle. Returns (warped, H)."""
    src = np.float32(corners4)
    dst = np.float32([[0, 0], [out_w, 0], [out_w, out_h], [0, out_h]])
    H = cv2.getPerspectiveTransform(src, dst)       # 3x3, bottom row [g, h, 1]
    return cv2.warpPerspective(img, H, (out_w, out_h)), H


def paste_into_quad(src, dst_img, quad4):
    """Warp src onto 4 corners in dst_img (e.g. painting into an angled frame) and paste it there."""
    h, w = src.shape[:2]
    H = cv2.getPerspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]]), np.float32(quad4))
    size = (dst_img.shape[1], dst_img.shape[0])
    warped = cv2.warpPerspective(src, H, size)
    mask = cv2.warpPerspective(np.full((h, w), 255, np.uint8), H, size)
    out = dst_img.copy()
    out[mask > 0] = warped[mask > 0]
    return out


def stitch_4points(left, right, pts_right4, pts_left4):
    """Lab 3 panorama with 4 hand-picked matching points: homography right -> left, warp, paste left."""
    H = cv2.getPerspectiveTransform(np.float32(pts_right4), np.float32(pts_left4))
    hL, wL = left.shape[:2]
    pano = cv2.warpPerspective(right, H, (wL + right.shape[1], max(hL, right.shape[0])))
    pano[:hL, :wL] = left
    return pano

## 4. Lab 4 — Filtering, gradients, edges, feature descriptors

In [ ]:
KERNELS = {
    'box':       np.ones((3, 3), np.float32) / 9,                                   # smoothing, sum 1
    'sharpen':   np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32),       # sum 1
    'emboss':    np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], np.float32),
    'laplacian': np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32),          # sum 0 -> edges
    'sobel_x':   np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32),
    'sobel_y':   np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], np.float32),
    'prewitt_x': np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], np.float32),
    'scharr_x':  np.array([[-3, 0, 3], [-10, 0, 10], [-3, 0, 3]], np.float32),
}


def apply_kernel(g, k, true_convolution=False):
    """filter2D with any kernel (CORRELATION by default; flips the kernel for true convolution).
    Computes in float so negative responses survive, then returns |result| as uint8."""
    if true_convolution:
        k = cv2.flip(k, -1)
    return cv2.convertScaleAbs(cv2.filter2D(g, cv2.CV_64F, k))


def out_size(N, K, P=0, S=1):
    """Output size of a convolution: floor((N - K + 2P)/S) + 1."""
    return (N - K + 2 * P) // S + 1


def conv_modes(g, k):
    """Return (same, valid, stride2) as float32: zero padding, border cropped, every 2nd pixel (filter2D has no stride)."""
    p = k.shape[0] // 2
    same = cv2.filter2D(g, cv2.CV_32F, k, borderType=cv2.BORDER_CONSTANT)   # float: keeps negatives
    return same, same[p:-p, p:-p], same[::2, ::2]          # use cv2.convertScaleAbs(...) to display


def add_salt_pepper(g, amount=0.03, seed=1):
    """Add salt-and-pepper noise to a gray image."""
    out = g.copy()
    r = np.random.default_rng(seed).random(g.shape)
    out[r < amount] = 0
    out[r > 1 - amount] = 255
    return out


def smooth_all(g, k=5):
    """Return dict of mean, Gaussian, median, bilateral results (median is best for salt & pepper)."""
    return {'mean': cv2.blur(g, (k, k)), 'gaussian': cv2.GaussianBlur(g, (k, k), 0),
            'median': cv2.medianBlur(g, k), 'bilateral': cv2.bilateralFilter(g, 9, 75, 75)}


def sobel_gradients(g, ksize=3):
    """Sobel gx, gy (CV_64F), magnitude sqrt(gx^2+gy^2) and direction atan2(gy,gx) in degrees.
    Returns (gx, gy, magnitude_uint8, angle_deg)."""
    gx = cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=ksize)
    gy = cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=ksize)
    mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
    return gx, gy, cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8), ang


def canny(g, low, high, blur_k=5):
    """Canny: Gaussian blur -> gradient -> non-max suppression -> hysteresis (low, high)."""
    return cv2.Canny(cv2.GaussianBlur(g, (blur_k, blur_k), 1.4), low, high)


def canny_compare(g, pairs=((50, 100), (50, 200), (100, 300))):
    """Run Canny for several (low, high) pairs and show them with their edge-pixel counts."""
    outs = [canny(g, lo, hi) for lo, hi in pairs]
    show([g] + outs, ['Original'] + [f'{lo}/{hi}: {cv2.countNonZero(e)} px' for (lo, hi), e in zip(pairs, outs)],
         cols=len(outs) + 1)
    return outs


def laplacian(g, ksize=3):
    """Laplacian (2nd derivative, edges at zero-crossings, very noise-sensitive) as uint8 magnitude."""
    return cv2.convertScaleAbs(cv2.Laplacian(g, cv2.CV_64F, ksize=ksize))


def log_edges(g, k=5, sigma=1.4):
    """Laplacian of Gaussian: smooth first, then Laplacian (fixes noise sensitivity)."""
    return laplacian(cv2.GaussianBlur(g, (k, k), sigma))


def hog_features(g, orientations=9, cell=8, block=2):
    """HOG descriptor (gradient-orientation histograms per cell, block-normalised). Returns (vector, visualisation)."""
    feat, vis = hog(g, orientations=orientations, pixels_per_cell=(cell, cell),
                    cells_per_block=(block, block), block_norm='L2-Hys', visualize=True)
    return feat, cv2.normalize(vis, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)


def lbp_features(g, P=8, R=1):
    """Uniform LBP texture codes and their normalised histogram (P+2 bins). Returns (lbp_image, hist)."""
    lbp = local_binary_pattern(g, P, R, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, P + 3), density=True)
    return lbp, hist


def hsv_hist(bgr, bins=(30, 32)):
    """Normalised 2-D Hue x Saturation colour histogram, flattened (global, rotation-invariant feature)."""
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    h = cv2.calcHist([hsv], [0, 1], None, list(bins), [0, 180, 0, 256])
    return cv2.normalize(h, h).flatten()


def hist_similarity(f1, f2, method=cv2.HISTCMP_CORREL):
    """Compare two histograms (CORREL: 1 = identical)."""
    return cv2.compareHist(np.float32(f1), np.float32(f2), method)


def edge_direction_hist(g, bins=18, low=50, high=150):
    """HED: histogram of gradient angles (0-180) at Canny edge pixels only."""
    _, _, _, ang = sobel_gradients(g)
    e = canny(g, low, high)
    h, _ = np.histogram(ang[e > 0] % 180, bins=bins, range=(0, 180), density=True)
    return h


def gradient_hist(g, bins=18):
    """HIG: magnitude-weighted histogram of gradient angles (0-180) over all pixels."""
    gx = cv2.Sobel(g, cv2.CV_64F, 1, 0); gy = cv2.Sobel(g, cv2.CV_64F, 0, 1)
    mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
    h, _ = np.histogram(ang % 180, bins=bins, range=(0, 180), weights=mag)
    return h / (h.sum() + 1e-9)

## 5. Lab 5 — Segmentation

In [ ]:
def global_thresholds(g, Ts=(60, 100, 150), inv=False):
    """Binary threshold at several T values (pixel > T -> 255). Returns list of masks."""
    mode = cv2.THRESH_BINARY_INV if inv else cv2.THRESH_BINARY
    return [cv2.threshold(g, T, 255, mode)[1] for T in Ts]


def adaptive(g, block=15, C=5, method='gaussian'):
    """Adaptive threshold: local T = (Gaussian-weighted or mean) neighbourhood mean - C. block must be odd."""
    m = cv2.ADAPTIVE_THRESH_GAUSSIAN_C if method == 'gaussian' else cv2.ADAPTIVE_THRESH_MEAN_C
    return cv2.adaptiveThreshold(g, 255, m, cv2.THRESH_BINARY, block, C)


def adaptive_grid(g, blocks=(3, 15, 51), Cs=(2, 10)):
    """Run adaptive thresholding for every (method, blockSize, C). Returns (images, titles)."""
    imgs, names = [], []
    for meth in ('mean', 'gaussian'):
        for b in blocks:
            for C in Cs:
                imgs.append(adaptive(g, b, C, meth)); names.append(f'{meth} b={b} C={C}')
    return imgs, names


def otsu(g, blur=True, inv=False):
    """Otsu threshold (maximises between-class variance; assumes bimodal histogram). Returns (T, mask)."""
    if blur:
        g = cv2.GaussianBlur(g, (5, 5), 0)
    mode = (cv2.THRESH_BINARY_INV if inv else cv2.THRESH_BINARY) + cv2.THRESH_OTSU
    return cv2.threshold(g, 0, 255, mode)


def plot_otsu(g):
    """Show original, histogram with the Otsu threshold line, and the Otsu mask."""
    T, m = otsu(g)
    plt.figure(figsize=(15, 4))
    plt.subplot(1, 3, 1); plt.imshow(g, cmap='gray'); plt.title('Original'); plt.axis('off')
    plt.subplot(1, 3, 2); plt.hist(g.ravel(), 256, range=(0, 256)); plt.axvline(T, color='r', ls='--')
    plt.title(f'Histogram, Otsu T={T:.0f}')
    plt.subplot(1, 3, 3); plt.imshow(m, cmap='gray'); plt.title('Otsu mask'); plt.axis('off')
    plt.tight_layout(); plt.show()
    return T, m


HSV_RANGES = {                      # OpenCV hue = degrees/2 (0-179)
    'red1': ((0, 100, 100), (10, 255, 255)), 'red2': ((170, 100, 100), (179, 255, 255)),
    'orange': ((10, 100, 100), (20, 255, 255)), 'yellow': ((20, 100, 100), (35, 255, 255)),
    'green': ((35, 80, 60), (85, 255, 255)), 'blue': ((100, 100, 60), (130, 255, 255)),
    'purple': ((130, 80, 60), (160, 255, 255)),
}


def hsv_mask(img, lower, upper, clean=True):
    """Binary mask of pixels inside an HSV box (inRange), optionally cleaned by opening. Returns (mask, extracted)."""
    img = to_bgr(img)                                   # gray input would crash cvtColor
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    m = cv2.inRange(hsv, np.array(lower), np.array(upper))
    if clean:
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    return m, cv2.bitwise_and(img, img, mask=m)


def colour_mask(img, name):
    """Mask by colour name from HSV_RANGES; 'red' combines both hue ends (red wraps around 0/179)."""
    if name == 'red':
        m1, _ = hsv_mask(img, *HSV_RANGES['red1'])
        m2, _ = hsv_mask(img, *HSV_RANGES['red2'])
        m = m1 | m2
        return m, cv2.bitwise_and(img, img, mask=m)
    return hsv_mask(img, *HSV_RANGES[name])


def region_growing(g, seed, T):
    """Grow from seed (x, y), adding 8-neighbours with |I - I_seed| <= T. Returns a 0/255 mask."""
    h, w = g.shape
    mask = np.zeros((h, w), np.uint8)
    sv = int(g[seed[1], seed[0]])                     # int: uint8 subtraction would wrap
    q = deque([seed]); mask[seed[1], seed[0]] = 255
    while q:
        x, y = q.popleft()
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                nx, ny = x + dx, y + dy
                if 0 <= nx < w and 0 <= ny < h and mask[ny, nx] == 0 and abs(int(g[ny, nx]) - sv) <= T:
                    mask[ny, nx] = 255
                    q.append((nx, ny))
    return mask


def watershed(img, dist_frac=0.7, inv=None):
    """Marker-based watershed for touching objects. Returns dict of every stage + 'count' + 'result' (red borders).
    inv=None -> automatic: if most of the border is white after Otsu the objects are dark, so it inverts for you.
    Stages: Otsu -> opening -> sure_bg (dilate) -> distance transform -> sure_fg -> unknown -> markers -> watershed."""
    img = to_bgr(img)
    g = cv2.GaussianBlur(to_gray(img), (5, 5), 0)
    _, th = otsu(g, blur=False, inv=bool(inv))                 # objects must be WHITE
    if inv is None:                                            # auto-polarity: background touches the border
        border = np.concatenate([th[0], th[-1], th[:, 0], th[:, -1]])
        if border.mean() > 127:
            th = cv2.bitwise_not(th)
    k = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(th, cv2.MORPH_OPEN, k, iterations=2)
    sure_bg = cv2.dilate(opening, k, iterations=3)
    dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, dist_frac * dist.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    unknown = cv2.subtract(sure_bg, sure_fg)
    n, markers = cv2.connectedComponents(sure_fg)               # int32 labels
    markers = markers + 1                                       # background 1, not 0
    markers[unknown == 255] = 0                                 # 0 = let watershed decide
    markers = cv2.watershed(img.copy(), markers)                # borders = -1
    result = img.copy(); result[markers == -1] = (0, 0, 255)
    return {'threshold': th, 'opening': opening, 'sure_bg': sure_bg, 'dist': dist, 'sure_fg': sure_fg,
            'unknown': unknown, 'markers': markers, 'fg_markers': n - 1,
            'count': len(np.unique(markers)) - 2, 'result': result}


def watershed_sweep(img, fracs=(0.3, 0.5, 0.7, 0.9)):
    """Run watershed at several distance-transform thresholds; returns a DataFrame of markers/regions."""
    rows = []
    for f in fracs:
        r = watershed(img, f)
        rows.append({'dist threshold': f'{f}*max', 'FG markers': r['fg_markers'], 'regions': r['count']})
    return pd.DataFrame(rows)


def kmeans_segment(img, K, attempts=10):
    """K-means colour segmentation: pixels as float32 (N,3) vectors, each replaced by its cluster centre.
    Returns (segmented_image, labels, centers, compactness)."""
    Z = img.reshape(-1, 1 if img.ndim == 2 else 3).astype(np.float32)   # gray -> 1 feature, BGR -> 3
    crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    comp, labels, centers = cv2.kmeans(Z, K, None, crit, attempts, cv2.KMEANS_RANDOM_CENTERS)
    centers = np.uint8(centers)
    return centers[labels.flatten()].reshape(img.shape), labels, centers, comp   # works for gray and BGR


def count_objects(mask, min_area=200):
    """Count blobs in a binary mask with findContours, ignoring tiny ones. Returns (count, contours)."""
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = [c for c in cnts if cv2.contourArea(c) > min_area]
    return len(cnts), cnts


def draw_boxes(img, contours, color=(0, 255, 0)):
    """Draw numbered bounding boxes around contours on a BGR copy of img."""
    out = to_bgr(img).copy()
    for i, c in enumerate(contours):
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(out, (x, y), (x + w, y + h), color, 2)
        cv2.putText(out, str(i + 1), (x, max(y - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    return out

## 6. Lab 6 — Hough, SIFT, panorama, wavelets, boundary detection

In [ ]:
def hough_circles(gray, dp=1.2, min_dist=60, param1=100, param2=30, min_r=20, max_r=80):
    """Detect circles on a median-blurred gray image. Lower param2 -> more (possibly false) circles.
    Returns (circles Nx3 [x, y, r] or empty array, drawn BGR image)."""
    c = cv2.HoughCircles(cv2.medianBlur(gray, 5), cv2.HOUGH_GRADIENT, dp=dp, minDist=min_dist,
                         param1=param1, param2=param2, minRadius=min_r, maxRadius=max_r)
    out = to_bgr(gray).copy()
    if c is None:                                      # None when nothing is found
        return np.empty((0, 3), np.uint16), out
    c = np.uint16(np.around(c[0]))
    for x, y, r in c:
        cv2.circle(out, (int(x), int(y)), int(r), (0, 255, 0), 2)
        cv2.circle(out, (int(x), int(y)), 2, (0, 0, 255), 3)
    cv2.putText(out, f'Count: {len(c)}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
    return c, out


def roi_trapezoid(edges, top_frac=0.5, top_width=0.1):
    """Keep only a lower trapezoid of an edge image (road region for lane detection)."""
    h, w = edges.shape
    poly = np.array([[(0, h), (w, h), (int(w * (0.5 + top_width / 2)), int(h * top_frac)),
                      (int(w * (0.5 - top_width / 2)), int(h * top_frac))]], np.int32)
    m = np.zeros_like(edges); cv2.fillPoly(m, poly, 255)
    return cv2.bitwise_and(edges, m)


def hough_lines_p(img, low=50, high=150, threshold=50, min_len=40, max_gap=20, roi=False, keep='all'):
    """Probabilistic Hough on Canny edges. keep: 'all', 'lanes' (|slope| > 0.3) or 'hv' (near horizontal/vertical,
    e.g. screen edges). Returns (segments list of (x1,y1,x2,y2), drawn image)."""
    e = canny(to_gray(img), low, high)
    if roi:
        e = roi_trapezoid(e)
    lines = cv2.HoughLinesP(e, 1, np.pi / 180, threshold, minLineLength=min_len, maxLineGap=max_gap)
    out, kept = to_bgr(img).copy(), []
    if lines is not None:
        for x1, y1, x2, y2 in lines[:, 0]:
            ang = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
            if keep == 'lanes' and (ang < 17 or ang > 163):          # drop near-horizontal
                continue
            if keep == 'hv' and not (ang < 10 or ang > 170 or abs(ang - 90) < 10):
                continue
            kept.append((x1, y1, x2, y2))
            cv2.line(out, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 3)
    return kept, out


def hough_lines_std(img, threshold=80, low=50, high=150):
    """Standard Hough (rho = x cos(theta) + y sin(theta)): infinite lines. Returns (list of (rho, theta), drawn image)."""
    e = canny(to_gray(img), low, high)
    lines = cv2.HoughLines(e, 1, np.pi / 180, threshold)
    out = to_bgr(img).copy()
    res = [] if lines is None else [tuple(l) for l in lines[:, 0]]
    for rho, th in res:
        a, b = np.cos(th), np.sin(th)
        x0, y0 = a * rho, b * rho
        cv2.line(out, (int(x0 - 1000 * b), int(y0 + 1000 * a)), (int(x0 + 1000 * b), int(y0 - 1000 * a)), (255, 0, 0), 1)
    return res, out


def sift_match(img1, img2, ratio=0.75):
    """SIFT keypoints/descriptors in both images, BF matching (L2) + Lowe's ratio test.
    Returns (kp1, kp2, good_matches). img1 is the query (indexed by queryIdx)."""
    sift = cv2.SIFT_create()
    kp1, d1 = sift.detectAndCompute(to_gray(img1), None)
    kp2, d2 = sift.detectAndCompute(to_gray(img2), None)
    if d1 is None or d2 is None:
        return kp1, kp2, []
    knn = cv2.BFMatcher(cv2.NORM_L2).knnMatch(d1, d2, k=2)
    good = [p[0] for p in knn if len(p) == 2 and p[0].distance < ratio * p[1].distance]
    return kp1, kp2, good


def sift_detect(ref, scene, min_matches=10, ratio=0.75):
    """Find ref inside scene: SIFT matches -> RANSAC homography -> draw the projected box.
    Returns (found, drawn_scene, matches_visualisation, n_good). found needs >= min_matches RANSAC inliers."""
    kp1, kp2, good = sift_match(ref, scene, ratio)
    out = to_bgr(scene).copy()
    vis = cv2.drawMatches(to_bgr(ref), kp1, out, kp2, good[:40], None,
                          flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    if len(good) < min_matches:
        return False, out, vis, len(good)
    src = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, inl = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    if H is None or int(inl.sum()) < min_matches:       # too few geometrically consistent matches
        return False, out, vis, len(good)
    h, w = ref.shape[:2]
    box = cv2.perspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2), H)
    cv2.polylines(out, [np.int32(box)], True, (0, 255, 0), 3)
    return True, out, vis, len(good)


def stitch(left, right, ratio=0.75):
    """SIFT panorama: match right->left, RANSAC homography, warp right onto a wide canvas, paste left, crop black.
    Returns (panorama, n_good)."""
    kpR, kpL, good = sift_match(right, left, ratio)          # query = right (the image that moves)
    if len(good) < 4:
        raise ValueError('Need >= 4 matches for a homography')
    pR = np.float32([kpR[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    pL = np.float32([kpL[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, _ = cv2.findHomography(pR, pL, cv2.RANSAC, 5.0)
    hL, wL = left.shape[:2]
    pano = cv2.warpPerspective(right, H, (wL + right.shape[1], max(hL, right.shape[0])))
    pano[:hL, :wL] = left
    cols = np.where(pano.max(axis=(0, 2)) > 0)[0]
    return pano[:, :cols[-1] + 1], len(good)


def wavelet_denoise(signal, wavelet='db4', level=4, mode='soft'):
    """DWT denoising: decompose, threshold detail coeffs with the universal threshold sigma*sqrt(2 ln N), reconstruct."""
    coeffs = pywt.wavedec(signal, wavelet, level=level)            # [cA_n, cD_n, ..., cD1]
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    T = sigma * np.sqrt(2 * np.log(len(signal)))
    coeffs = [coeffs[0]] + [pywt.threshold(c, T, mode=mode) for c in coeffs[1:]]
    return pywt.waverec(coeffs, wavelet)[:len(signal)]             # may be 1 sample longer


def wavelet_anomalies(signal, k=3, **kw):
    """Anomalies = points where |signal - denoised| > k*std(residual). Returns (indices, denoised, residual)."""
    d = wavelet_denoise(signal, **kw)
    res = signal - d
    return np.where(np.abs(res) > k * res.std())[0], d, res


def plot_anomalies(signal, denoised, residual, idx, k=3):
    """Two-panel plot: signal vs denoised, and residuals with anomalies marked (Lab 6 expected output)."""
    plt.figure(figsize=(12, 7))
    plt.subplot(2, 1, 1); plt.plot(signal, alpha=0.6, label='Sensor data')
    plt.plot(denoised, 'r--', label='Denoised'); plt.legend(); plt.title('Sensor Data and Denoised Signal')
    plt.subplot(2, 1, 2); plt.plot(residual, label='Residuals')
    plt.scatter(idx, residual[idx], color='r', zorder=3, label='Anomalies')
    for s in (1, -1):
        plt.axhline(s * k * residual.std(), color='k', ls=':')
    plt.legend(); plt.title('Residuals and Detected Anomalies'); plt.tight_layout(); plt.show()


def dwt2_bands(g, wavelet='haar'):
    """2-D DWT: returns LL (approximation), LH (horizontal), HL (vertical), HH (diagonal), each half size."""
    LL, (LH, HL, HH) = pywt.dwt2(g.astype(np.float64), wavelet)
    return LL, LH, HL, HH


def security_zone(frame, zone, background, min_area=500, diff_T=30):
    """Find NEW objects (pixels that differ from an empty-scene `background` frame) and flag any whose
    centre lies inside zone=(x1,y1,x2,y2). Returns (alarm, drawn_frame).
    Video loop: background = first frame (or cap.read() of an empty scene); call once per frame.
    (Otsu alone is wrong here: it always splits the image in two, so furniture/floor would trigger alarms.)"""
    diff = cv2.absdiff(to_gray(frame), to_gray(background))
    _, m = cv2.threshold(cv2.GaussianBlur(diff, (5, 5), 0), diff_T, 255, cv2.THRESH_BINARY)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = to_bgr(frame).copy()
    cv2.rectangle(out, zone[:2], zone[2:], (255, 0, 0), 2)
    alarm = False
    for c in cnts:
        if cv2.contourArea(c) < min_area:
            continue
        x, y, w, h = cv2.boundingRect(c)
        cx, cy = x + w // 2, y + h // 2
        inside = zone[0] <= cx <= zone[2] and zone[1] <= cy <= zone[3]
        cv2.drawContours(out, [c], -1, (0, 0, 255) if inside else (0, 255, 0), 2)
        alarm |= inside
    if alarm:
        cv2.putText(out, 'ALARM', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    return alarm, out


def security_video(path, zone, background=None, show_live=True):
    """Real-time loop: first frame = background unless given. Press q to quit. Returns frames that raised an alarm."""
    cap = cv2.VideoCapture(path)
    alarms, i = [], 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if background is None:
            background = frame.copy()
        alarm, out = security_zone(frame, zone, background)
        if alarm:
            alarms.append(i)
        if show_live:
            cv2.imshow('Security', out)
            if cv2.waitKey(30) & 0xFF == ord('q'):
                break
        i += 1
    cap.release()
    if show_live:
        cv2.destroyAllWindows()
    return alarms


## Index — function → what it does
| Lab | Functions |
|---|---|
| Setup | `show`, `load`, `to_gray`, `to_bgr`, `make_*` practice images |
| 1 | `show_rgb`, `resize`, `crop_center`, `gaussian_blur`, `draw_target`, `caption_overlay`, `mask_combine`, `channel_stats`, `rotate` |
| 2 | `negative`, `log_transform`, `inverse_log_transform`, `gamma_transform`, `contrast_stretch`, `minmax_stretch`, `slice_levels`, `bit_plane`, `plot_hist`, `equalize`, `equalize_manual`, `clahe`, `equalize_colour`, `compare_with_hist`, `plot_transfer_curves`, `colormap`, `gray_world`, `brightness_contrast`, `dense_mask`, `fuse`, `enhance_frame`, `process_video` |
| 3 | `T`, `S`, `Sh`, `R`, `about_center`, `apply`, `scale_centered`, `rotate_no_clip`, `shear_no_clip`, `rigid`, `similarity`, `solve_affine`, `birdseye`, `paste_into_quad`, `stitch_4points` |
| 4 | `KERNELS`, `apply_kernel`, `out_size`, `conv_modes`, `add_salt_pepper`, `smooth_all`, `sobel_gradients`, `canny`, `canny_compare`, `laplacian`, `log_edges`, `hog_features`, `lbp_features`, `hsv_hist`, `hist_similarity`, `edge_direction_hist`, `gradient_hist` |
| 5 | `global_thresholds`, `adaptive`, `adaptive_grid`, `otsu`, `plot_otsu`, `HSV_RANGES`, `hsv_mask`, `colour_mask`, `region_growing`, `watershed`, `watershed_sweep`, `kmeans_segment`, `count_objects`, `draw_boxes` |
| 6 | `hough_circles`, `roi_trapezoid`, `hough_lines_p`, `hough_lines_std`, `sift_match`, `sift_detect`, `stitch`, `wavelet_denoise`, `wavelet_anomalies`, `plot_anomalies`, `dwt2_bands`, `security_zone` (needs a background frame), `security_video` |

## Self-test — runs every section on practice images (also shows example usage)

In [ ]:
# Lab 1
img = make_colour_objects()
canvas, centre = draw_target()
print('canvas centre:', centre)
_, desc = channel_stats(img)
show([canvas, caption_overlay(img, 'Title'), mask_combine(img, make_grid_img())[3], rotate(img, 45, 0.8)],
     ['Target', 'Caption', 'Mask combine', 'Rotate 45'], cols=4)

# Lab 2
x = make_dark_xray()
show([x, log_transform(x), gamma_transform(x, 0.5), contrast_stretch(x, int(x.min()), 0, int(x.max()), 255),
      equalize(x), clahe(x)], ['Raw', 'Log', 'Gamma 0.5', 'Stretch', 'Equalized', 'CLAHE'])
print('manual vs builtin HE max diff:', np.abs(equalize(x).astype(int) - equalize_manual(x).astype(int)).max())
plot_transfer_curves()

# Lab 3
gimg = make_grid_img()
src3 = np.float32([[50, 50], [350, 50], [50, 250]]); dst3 = np.float32([[70, 80], [330, 40], [110, 260]])
print('affine manual == cv2:', np.allclose(solve_affine(src3, dst3), cv2.getAffineTransform(src3, dst3), atol=1e-4))
show([scale_centered(gimg, 3), rotate_no_clip(gimg, 45), shear_no_clip(gimg, -0.3),
      birdseye(gimg, [[150, 60], [250, 60], [340, 250], [60, 250]], 300, 300)[0]],
     ['Scale x3', 'Rotate 45 no clip', 'Shear', "Bird's-eye"], cols=4)

# Lab 4
g = to_gray(img)
print('out_size(32,5,0,1) =', out_size(32, 5, 0, 1))
_, _, mag, _ = sobel_gradients(g)
if hog is not None:
    print('HOG length:', hog_features(g)[0].shape[0], '| LBP bins:', len(lbp_features(g)[1]))
show([apply_kernel(g, KERNELS['sharpen']), mag, canny(g, 50, 150), log_edges(g)],
     ['Sharpen', 'Sobel magnitude', 'Canny', 'LoG'], cols=4)

# Lab 5
T_, _ = otsu(make_coins())
ws = watershed(make_coins(), 0.7)
seg, *_ = kmeans_segment(img, 4)
print('Otsu T:', T_, '| watershed count:', ws['count'])
print(watershed_sweep(make_coins()).to_string(index=False))
show([adaptive(make_document()), colour_mask(img, 'yellow')[1], seg, ws['result'],
      region_growing(cv2.GaussianBlur(x, (5, 5), 0), (100, 150), 15)],
     ['Adaptive', 'Yellow mask', 'K-means K=4', 'Watershed', 'Region growing'], cols=5)

# Lab 6
circles, circ_img = hough_circles(make_coins(touching=False))
segs, lane_img = hough_lines_p(make_road(), roi=True, keep='lanes')
scene = make_textured()
test = cv2.warpAffine(scene, cv2.getRotationMatrix2D((300, 200), 20, 0.8), (600, 400), borderValue=(200, 200, 200))
found, det_img, _, n_good = sift_detect(scene[100:250, 200:380], test)
full = make_textured(400, 900, 3)
pano, _ = stitch(full[:, :550], full[:, 350:])
print('coins:', len(circles), '| lane segments:', len(segs), '| SIFT found:', found, n_good, '| pano:', pano.shape)
show([circ_img, lane_img, det_img, pano], ['Hough circles', 'Lanes', 'SIFT detection', 'Panorama'], cols=4)
if pywt is not None:
    sig, true_idx = make_sensor_signal()
    idx, d, res = wavelet_anomalies(sig)
    print('anomalies:', idx.tolist(), '| injected:', true_idx)
    plot_anomalies(sig, d, res, idx)
empty = make_colour_objects()
intruder = empty.copy(); cv2.rectangle(intruder, (180, 100), (260, 200), (30, 30, 30), -1)
print('security alarm | empty scene:', security_zone(empty, (150, 80, 300, 220), empty)[0],
      '| intruder inside zone:', security_zone(intruder, (150, 80, 300, 220), empty)[0])